# Estatísticas por Copa e por seleção

## Task 3: calcular as estatísticas por Copa e por seleção

## O que a transmissão precisa

1. **Por Copa:** partidas, gols, média de gols por partida e público médio.
2. **Por seleção:** jogos, vitórias, gols marcados, gols sofridos e saldo.

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-estatisticas")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta: é o mesmo código dos vídeos 2.1 (leitura e limpeza) e 2.2 (padronização), na mesma
ordem.

In [2]:
# vídeo 2.1: leitura e limpeza
from wc_pipeline.schemas import MATCHES

partidas_brutas = (
    spark.read
    .option("header", "true")
    .schema(MATCHES)
    .csv(f"{RAW}/WorldCupMatches.csv")
)

partidas_limpas = (
    partidas_brutas
    .filter(F.col("MatchID").isNotNull())
    .dropDuplicates(["MatchID"])
)

# vídeo 2.2: padronização
partidas = partidas_limpas.select(
    F.col("Year").alias("ano"),
    F.col("MatchID").alias("match_id"),
    F.col("Stage").alias("fase"),
    F.col("Home Team Name").alias("mandante"),
    F.col("Home Team Goals").alias("gols_mandante"),
    F.col("Away Team Goals").alias("gols_visitante"),
    F.col("Away Team Name").alias("visitante"),
    F.col("Attendance").alias("publico"),
    F.col("Datetime").alias("data_partida"),
)

partidas = partidas.withColumn(
    "publico",
    F.regexp_replace("publico", ",", "").cast("int"),
)

FORMATO_DATA = "dd MMM yyyy - HH:mm"

partidas = partidas.withColumn(
    "data_partida",
    F.to_timestamp("data_partida", FORMATO_DATA),
)

partidas = (
    partidas
    .withColumn("total_gols", F.col("gols_mandante") + F.col("gols_visitante"))
    .withColumn(
        "resultado",
        F.when(F.col("gols_mandante") > F.col("gols_visitante"), "mandante")
         .when(F.col("gols_mandante") < F.col("gols_visitante"), "visitante")
         .otherwise("empate"),
    )
)

partidas.printSchema()

root
 |-- ano: integer (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- fase: string (nullable = true)
 |-- mandante: string (nullable = true)
 |-- gols_mandante: integer (nullable = true)
 |-- gols_visitante: integer (nullable = true)
 |-- visitante: string (nullable = true)
 |-- publico: integer (nullable = true)
 |-- data_partida: timestamp (nullable = true)
 |-- total_gols: integer (nullable = true)
 |-- resultado: string (nullable = false)



## Agregar sem agrupar: o histórico inteiro

O resumo de todas as Copas, a referência para comparar cada Copa com a média histórica. As funções de agregação
(`F.count`, `F.sum`, `F.avg`, `F.max`...) dentro de um `select` resumem a tabela numa linha.

In [3]:
partidas.select(
    F.count("*").alias("partidas"),
    F.sum("total_gols").alias("gols"),
    F.round(F.avg("total_gols"), 2).alias("media_gols"),
    F.max("total_gols").alias("maior_placar"),
).show()

+--------+----+----------+------------+
|partidas|gols|media_gols|maior_placar|
+--------+----+----------+------------+
|     562|2542|      4.52|           9|
+--------+----+----------+------------+



`F.round(..., 2)` arredonda a média para duas casas.

## groupBy + agg: uma linha por Copa

`groupBy` separa os grupos; `agg` diz o que calcular em cada um.

In [4]:
#  partidas, gols, média de gols por partida e público médio
por_copa = (
    partidas.groupBy("ano")
    .agg(
        F.count("*").alias("partidas"),
        F.sum("total_gols").alias("gols"),
        F.round(F.avg("total_gols"), 2).alias("media_gols"),
        F.round(F.avg("publico"), 0).alias("publico_medio"),
    )
    .orderBy("ano")
)
por_copa.show(20)

+----+--------+----+----------+-------------+
| ano|partidas|gols|media_gols|publico_medio|
+----+--------+----+----------+-------------+
|1950|      33| 128|      3.88|      55669.0|
|1954|      25| 124|      4.96|      57877.0|
|1958|      34| 156|      4.59|      52685.0|
|1962|      27| 129|      4.78|      52304.0|
|1966|      25| 123|      4.92|      57402.0|
|1970|      26| 111|      4.27|      52217.0|
|1974|      33| 152|      4.61|      49791.0|
|1978|      24| 109|      4.54|      44263.0|
|1982|      30| 143|      4.77|      58046.0|
|1986|      39| 171|      4.38|      46923.0|
|1990|      33| 152|      4.61|      55693.0|
|1994|      39| 176|      4.51|      54808.0|
|1998|      33| 137|      4.15|      53962.0|
|2002|      30| 143|      4.77|      56599.0|
|2006|      28| 124|      4.43|      52876.0|
|2010|      28| 114|      4.07|      50985.0|
|2014|      36| 166|      4.61|      58937.0|
|2018|      39| 184|      4.72|      52249.0|
+----+--------+----+----------+---

`avg`, `sum` e as outras agregações **ignoram nulos**; só o `count("*")` conta todas as linhas.

## Por seleção: o truque de virar a partida

Cada partida tem duas seleções, em colunas diferentes. Mudamos o **grão** (o que cada linha representa): cada partida
vira duas linhas, uma por seleção.

```
partida:   Brazil  3 x 1  Mexico

vira:      selecao   gols_pro   gols_contra
           Brazil       3           1
           Mexico       1           3
```

In [17]:
partidas.filter(F.col("mandante") == "England").show(2)
partidas.filter(F.col("visitante") == "England").show(2)

+----+--------+-----------+--------+-------------+--------------+---------+-------+-------------------+----------+---------+
| ano|match_id|       fase|mandante|gols_mandante|gols_visitante|visitante|publico|       data_partida|total_gols|resultado|
+----+--------+-----------+--------+-------------+--------------+---------+-------+-------------------+----------+---------+
|1950|    1001|Round of 16| England|            2|             1|    Italy|  62576|1950-06-13 12:00:00|         3| mandante|
|1950|    1003|    Group A| England|            2|             3| Portugal|  23392|1950-06-15 19:00:00|         5|visitante|
+----+--------+-----------+--------+-------------+--------------+---------+-------+-------------------+----------+---------+
only showing top 2 rows
+----+--------+-------+-----------+-------------+--------------+---------+-------+-------------------+----------+---------+
| ano|match_id|   fase|   mandante|gols_mandante|gols_visitante|visitante|publico|       data_partida|

In [10]:
mandante = partidas.select(
    "ano",
    F.col("mandante").alias("selecao"),
    F.col("gols_mandante").alias("gols_pro"),
    F.col("gols_visitante").alias("gols_contra"),
)
visitante = partidas.select(
    "ano",
    F.col("visitante").alias("selecao"),
    F.col("gols_visitante").alias("gols_pro"),
    F.col("gols_mandante").alias("gols_contra"),
)

por_selecao = mandante.unionByName(visitante)
print("partidas:", partidas.count(), "| linhas por seleção:", por_selecao.count())

partidas: 562 | linhas por seleção: 1124


In [18]:
por_selecao.filter(F.col("selecao") == "England").show(4)

+----+-------+--------+-----------+
| ano|selecao|gols_pro|gols_contra|
+----+-------+--------+-----------+
|1950|England|       2|          1|
|1950|England|       2|          3|
|1950|England|       3|          0|
|1950|England|       4|          4|
+----+-------+--------+-----------+
only showing top 4 rows


`union` empilha **pela posição**; `unionByName`, **pelo nome**, que é o mais seguro.

In [9]:
# jogos, vitórias, gols marcados, gols sofridos e saldo

por_selecao = por_selecao.withColumn(
    "resultado",
    F.when(F.col("gols_pro") > F.col("gols_contra"), "V")
     .when(F.col("gols_pro") < F.col("gols_contra"), "D")
     .otherwise("E"),
)

tabela_selecoes = (
    por_selecao.groupBy("selecao")
    .agg(
        F.count("*").alias("jogos"),
        F.sum(F.when(F.col("resultado") == "V", 1).otherwise(0)).alias("vitorias"),
        F.sum("gols_pro").alias("gols_pro"),
        F.sum("gols_contra").alias("gols_contra"),
    )
    .withColumn("saldo", F.col("gols_pro") - F.col("gols_contra"))
    .orderBy(F.desc("saldo"))
)
tabela_selecoes.show(16)

+-----------+-----+--------+--------+-----------+-----+
|    selecao|jogos|vitorias|gols_pro|gols_contra|saldo|
+-----------+-----+--------+--------+-----------+-----+
|     Brazil|   73|      38|     162|        116|   46|
|  Argentina|   63|      32|     142|        109|   33|
|    Nigeria|   76|      33|     165|        159|    6|
|Netherlands|   58|      27|     132|        128|    4|
|      Spain|   67|      31|     155|        153|    2|
|    Uruguay|   67|      27|     161|        159|    2|
|    England|   76|      33|     179|        178|    1|
|    Germany|   70|      29|     169|        169|    0|
|      Italy|   66|      30|     162|        163|   -1|
|    Belgium|   67|      32|     158|        160|   -2|
|     France|   85|      31|     194|        199|   -5|
|     Mexico|   75|      29|     168|        173|   -5|
|   Portugal|   72|      29|     160|        168|   -8|
|      Japan|   78|      22|     169|        188|  -19|
|    Croatia|   64|      22|     141|        164

`F.sum(F.when(...))` **conta com condição**: cada vitória vale 1, o resto 0. `F.desc` ordena do maior para o menor.

## Conferindo o resultado

O total de gols tem que ser o mesmo pelos dois caminhos: por Copa e por seleção. A ação `first()` traz a primeira
linha do resultado, e `[0]` pega o primeiro valor dela.

In [19]:
gols_por_copa = por_copa.select(F.sum("gols")).first()[0]
gols_por_selecao = tabela_selecoes.select(F.sum("gols_pro")).first()[0]

print("soma dos gols por Copa   :", gols_por_copa)
print("soma dos gols por seleção:", gols_por_selecao)

soma dos gols por Copa   : 2542
soma dos gols por seleção: 2542


## Para consultar depois

- **`groupBy("ano").count()`** é um atalho que só conta, com a coluna chamada `count`. O `agg` permite várias
  métricas, com os nomes que você escolher.
- **Outras agregações úteis:** `F.countDistinct("estadio")` (quantos valores diferentes), `F.min`, `F.max` e
  `F.stddev`. `F.collect_set` e `F.collect_list` juntam os valores numa lista; cuidado em dado grande, porque uma chave
  com milhões de linhas vira uma lista gigante numa linha só.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [20]:
spark.stop()